# GRPO Training Experiment

> Historical experiment retained for reproducibility; not used by the production pipeline. Set `HF_TOKEN` and `WANDB_TOKEN` in Colab Secrets. The trained model is published to the Hugging Face account associated with `HF_TOKEN`.

## 1. Install Dependencies

In [1]:
!pip install -q unsloth
!pip install -q trl transformers datasets accelerate peft bitsandbytes
!pip install -q wandb huggingface_hub
!pip install -q rouge-score

  Preparing metadata (setup.py) ... done


# 2. Import + Login

In [2]:
import re
import torch
import wandb

from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_TOKEN = userdata.get("WANDB_TOKEN")
OUTPUT_MODEL_ID = f"{HfApi().whoami(token=HF_TOKEN)['name']}/legal-chatbot-qwen-grpo"

login(HF_TOKEN)
wandb.login(key=WANDB_TOKEN)

wandb.init(
    project="legal-chatbot-qwen-grpo",
    name="qwen25-3b-grpo"
)

# 3. Load Model

In [3]:
import torch
import re

from unsloth import FastLanguageModel, PatchFastRL

PatchFastRL()

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Slotherynn/legal-chatbot-qwen-sft",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platfor

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# 4. Format Reward Function

In [4]:

def format_reward_func(prompts, completions, **kwargs):

    rewards = []

    for completion in completions:

        text = (
            completion[0]["content"]
            if isinstance(completion, list)
            else completion
        )

        reward = 0.0

        if text.startswith("<think>"):
            reward += 0.2

        if "</think>" in text:
            reward += 0.3

        match = re.match(
            r"^<think>([\s\S]*?)</think>([\s\S]+)$",
            text.strip()
        )

        if match:
            reward += 0.5

        if (
            text.count("<think>") > 1 or
            text.count("</think>") > 1
        ):
            reward -= 0.5

        rewards.append(reward)

    return rewards

# 5. Reasoning Length Reward

In [5]:
def reasoning_length_reward(
    prompts,
    completions,
    **kwargs
):

    rewards = []

    for completion in completions:

        text = (
            completion[0]["content"]
            if isinstance(completion, list)
            else completion
        )

        match = re.search(
            r"<think>([\s\S]*?)</think>",
            text
        )

        if not match:
            rewards.append(0.0)
            continue

        think_content = match.group(1).strip()

        length = len(think_content)

        if length == 0:
            rewards.append(0.0)

        elif length < 50:
            rewards.append(0.2)

        elif length < 200:
            rewards.append(0.5)

        else:
            rewards.append(1.0)

    return rewards

# 6. Correctness & Language Reward

In [6]:
from rouge_score import rouge_scorer

def correctness_reward(
    prompts,
    completions,
    answer,
    **kwargs
):

    rewards = []

    scorer = rouge_scorer.RougeScorer(
        ['rougeL'],
        use_stemmer=True
    )

    for completion, ground_truth in zip(
        completions,
        answer
    ):

        text = (
            completion[0]["content"]
            if isinstance(completion, list)
            else completion
        )

        final_output = (
            text.split("</think>")[-1]
            .strip()
        )

        scores = scorer.score(
            ground_truth,
            final_output
        )

        rouge_l_score = (
            scores['rougeL'].fmeasure
        )

        rewards.append(rouge_l_score)

    return rewards


def language_reward_func(
    prompts,
    completions,
    **kwargs
):

    rewards = []

    english_indicators = [
        "Therefore",
        "However",
        "In conclusion",
        "According to",
        "The employee is"
    ]

    for completion in completions:

        text = (
            completion[0]["content"]
            if isinstance(completion, list)
            else completion
        )

        final_output = (
            text.split("</think>")[-1]
            .strip()
        )

        if any(
            word in final_output
            for word in english_indicators
        ):
            rewards.append(-0.5)

        else:
            rewards.append(1.0)

    return rewards

# 7. Dataset GRPO

In [7]:
from datasets import load_dataset

dataset_grpo = load_dataset(
    "Ichsan2895/alpaca-gpt4-indonesian",
    split="train[:2000]"
)

print(
    "Kolom Dataset:",
    dataset_grpo.column_names
)

def map_grpo_format(example):

    prompt_content = example.get(
        "instruction",
        example.get(
            "text",
            example.get(
                "prompt",
                ""
            )
        )
    )

    if (
        not prompt_content and
        example.get("input")
    ):
        prompt_content = example.get(
            "input"
        )

    elif example.get("input"):

        prompt_content = (
            f"{prompt_content}\n"
            f"Konteks Tambahan: "
            f"{example.get('input')}"
        )

    answer_content = example.get(
        "output",
        example.get(
            "response",
            ""
        )
    )

    return {
        "prompt": [
            {
                "role": "user",
                "content": prompt_content
            }
        ],
        "answer": answer_content
    }

dataset_grpo = dataset_grpo.map(
    map_grpo_format
)

print(dataset_grpo[0])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Kolom Dataset: ['Unnamed: 0', 'input', 'output']


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'Unnamed: 0': 1, 'input': 'Saranlah slogan untuk kampanye daur ulang\n', 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."', 'prompt': [{'role': 'user', 'content': 'Saranlah slogan untuk kampanye daur ulang\n'}], 'answer': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}


# 8. GRPO Config + Trainer

In [8]:
from trl import (
    GRPOTrainer,
    GRPOConfig
)

training_args = GRPOConfig(
    output_dir="outputs_qwen_grpo",

    learning_rate=5e-6,

    weight_decay=0.01,

    adam_beta1=0.9,
    adam_beta2=0.99,

    max_prompt_length=256,

    max_completion_length=512,

    num_generations=4,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    max_steps=200,

    logging_steps=1,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    report_to="wandb",

    save_strategy="no",

    use_vllm=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,

    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func
    ],

    args=training_args,

    train_dataset=dataset_grpo,
)

# 9. Training

In [9]:
trainer.train()
wandb.finish()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Futu

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
1,0.000001,1.158453,0.149917,62.250000,12.000000,124.000000,0.000000,62.250000,12.000000,124.000000,0.000008,0.000000,0.000000,0.000000,0.000000,0.158453,0.149917,1.000000,0.000000
2,0.000004,1.179900,0.021996,244.000000,142.000000,327.000000,0.000000,244.000000,142.000000,327.000000,0.000011,0.000000,0.000000,0.000000,0.000000,0.179900,0.021996,1.000000,0.000000
3,-0.000001,1.057851,0.023938,17.750000,11.000000,34.000000,0.000000,17.750000,11.000000,34.000000,0.000022,0.000000,0.000000,0.000000,0.000000,0.057851,0.023938,1.000000,0.000000
4,-0.000001,1.155788,0.042117,303.750000,174.000000,503.000000,0.000000,303.750000,174.000000,503.000000,0.000010,0.000000,0.000000,0.000000,0.000000,0.155788,0.042117,1.000000,0.000000
5,0.000001,1.225794,0.041064,290.750000,220.000000,391.000000,0.000000,290.750000,220.000000,391.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.225794,0.041064,1.000000,0.000000
6,0.000000,1.113390,0.004310,199.250000,183.000000,227.000000,0.000000,199.250000,183.000000,227.000000,0.000012,0.000000,0.000000,0.000000,0.000000,0.113390,0.004310,1.000000,0.000000
7,0.000001,1.223490,0.052883,344.250000,212.000000,467.000000,0.000000,344.250000,212.000000,467.000000,0.000008,0.000000,0.000000,0.000000,0.000000,0.223490,0.052883,1.000000,0.000000
8,-0.000001,1.477355,0.087969,97.500000,25.000000,194.000000,0.000000,97.500000,25.000000,194.000000,0.000007,0.000000,0.000000,0.000000,0.000000,0.477355,0.087969,1.000000,0.000000
9,0.000000,1.269083,0.017546,50.750000,32.000000,94.000000,0.000000,50.750000,32.000000,94.000000,0.000008,0.000000,0.000000,0.000000,0.000000,0.269083,0.017546,1.000000,0.000000
10,0.000000,1.219088,0.006207,462.750000,315.000000,512.000000,0.750000,315.000000,315.000000,315.000000,0.000009,0.000000,0.000000,0.000000,0.000000,0.219088,0.006207,1.000000,0.000000


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=5

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▃▃▃▂▂▁▂▁▂▁▁▁▁▃▃▁▂▁▁▁▃▁█▁▁▁▁▁▁▁▁▃▄▁▃▄▂▁▁▂
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▁▁▁▁▁▁▁▆▁█▁▁▁▄▁▁▁▃▁▁▃▁▁▁▁▅▁▁▁▁▁▁▁██▁▁▁▇
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,▃▃▁▁▅▁▄▁▃▁▁▃▃▂▁▁▄▇▄▁▂▁▄▁▄█▁▁▂▁▁▁▃▁▁▃▂▃▄▂
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▄▃▇▃▃▃▃▄▂▂▆▅▄█▃▁▃▄▃▄▃▃▃▂▂▅▃▂▃▁▇▂▂▄▃▅▃▂▂▃
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▅▂▂▂▂█▃▂▃▁▃▅▅▁▁▃▂▂▄▃▁▁▃▂▄▃▂▁▄▃▂▃▂▂▂▂▃▁▃▂
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▅▁█▂▂▂▁▁▃▂▂▂▄▂▂▂▃▂▁▁▂▂▂▂▃▁▁▂▂▂▃▁▂▂▂▃▃▃▂▂
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▄▇▂▅▁█▁█▃▆▇▄█▆▂▂▄▂▄▂█▂█▂▁██▂▂▁▄▂▃█▁▂█▅▆█
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+28,...


# 10. Push Model

In [ ]:
model.push_to_hub_merged(
    OUTPUT_MODEL_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN
)